In [1]:
! export HF_HOME="/scratch/yun.hy/.cache"
! export HUGGINGFACE_HUB_CACHE="/scratch/yun.hy/.cache"
! export XDG_CACHE_HOME="/scratch/yun.hy/.cache"

In [2]:
from utils import load_jsonl_file, load_json_file
import json
import spacy
import re
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
import textstat
from tqdm import tqdm

/home/yun.hy/.conda/envs/llm_health_framing_effect/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class Evaluator:
    def __init__(self):
        print("Loading models (this may take a minute on first run)...")
        # 1. High-accuracy Sentiment Model (SiEBERT)
        self.sentiment_pipe = pipeline(
            "sentiment-analysis", 
            model="siebert/sentiment-roberta-large-english"
        )
        
        # 2. Semantic Similarity Model
        self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        # 3. NLP for Entities
        self.nlp = spacy.load("en_core_web_sm")
        
        self.hedge_words = [
            "maybe", "probably", "possibly", "likely", "unlikely",
            "suggests", "appears", "seems", "might", "could", "may",
            "to some extent", "almost", "fairly", "quite", "rather",
            "I think", "in my opinion", "generally", "typically",
            "possibly", "potentially", "roughly", "tends to", "approximately",
            "more or less", "nearly"
        ]

    def get_entities(self, text):
        doc = self.nlp(text)
        return set([ent.text.lower() for ent in doc.ents])

    def get_roberta_sentiment(self, text):
        """Returns the label and confidence score from SiEBERT."""
        result = self.sentiment_pipe(text[:512])[0]  # Truncated to 512 for RoBERTa limits
        return {
            "label": result['label'],
            "confidence": round(result['score'], 4)
        }

    def evaluate_pair(self, text_a, text_b):
        # Semantic Similarity
        emb1 = self.semantic_model.encode(text_a, convert_to_tensor=True)
        emb2 = self.semantic_model.encode(text_b, convert_to_tensor=True)
        sem_sim = util.pytorch_cos_sim(emb1, emb2).item()

        # Entity Overlap
        ent_a = self.get_entities(text_a)
        ent_b = self.get_entities(text_b)
        intersection = ent_a.intersection(ent_b)
        ne_overlap = len(intersection) / len(ent_a.union(ent_b)) if ent_a.union(ent_b) else 1.0

        # Lexical Stats
        def get_stats(text):
            sent = self.get_roberta_sentiment(text)
            hedges = sum(1 for w in self.hedge_words if re.search(r'\b' + w + r'\b', text.lower()))
            return {
                "sentiment": sent['label'],
                "sentiment_confidence": sent['confidence'],
                "hedge_count": hedges,
                "reading_ease": textstat.flesch_reading_ease(text),
                "length_words": len(text.split())
            }

        return {
            "comparison": {
                "semantic_similarity": f"{sem_sim:.2%}",
                "entity_overlap": f"{ne_overlap:.2%}",
                "common_entities": list(intersection)
            },
            "response_a_metrics": get_stats(text_a),
            "response_b_metrics": get_stats(text_b)
        }

    def process_batch(self, input_data):
        results = {}
        for uid, pairs in tqdm(input_data.items(), desc="Processing Items"):
            if uid == "multiturn":
                continue
            # print(f"Analyzing {uid}...")
            results[uid] = self.evaluate_pair(pairs['positive_answer'], pairs['negative_answer'])
        return results

In [4]:
def format_outputs(raw_data):
    grouped = {}
    
    for item in raw_data:
        cid = item["custom_id"]
        text = item["output_text"]
            
        key = cid.split("-")[1]
        parts = key.split("_")
        
        # The last part is the sentiment (positive/negative)
        sentiment = parts[-1]
        
        # e.g., "CD002818_effectiveness1"
        pair_key = "_".join(parts[:-1])

        if pair_key not in grouped:
            grouped[pair_key] = {}

        if sentiment == "positive":
            grouped[pair_key]["positive_answer"] = text
        elif sentiment == "negative":
            grouped[pair_key]["negative_answer"] = text
            
    return grouped
        

In [5]:
file_to_load_gpt = "outputs/responses/gpt-5.1/batch_question_responses.jsonl"

data_gpt = load_jsonl_file(file_to_load_gpt)

data_gpt = format_outputs(data_gpt)
print(f"entries: {len(data_gpt.keys())}")

# Run
evaluator = Evaluator()
final_report_gpt = evaluator.process_batch(data_gpt)

print("\n--- FINAL EVALUATION REPORT for GPT 5.1 OUTPUTS---")
print(json.dumps(final_report_gpt, indent=2))

entries: 4416
Loading models (this may take a minute on first run)...


Device set to use cuda:0
Processing Items:   1%|          | 40/4416 [00:39<1:12:17,  1.01it/s]


KeyboardInterrupt: 

In [ ]:
file_to_load_claude = "outputs/responses/claude_4.5_sonnet/batch_question_responses.jsonl"

data_claude = load_jsonl_file(file_to_load_claude)

data_claude = format_outputs(data_claude)
print(f"entries: {len(data_claude.keys())}")

# Run
evaluator = Evaluator()
final_report_claude = evaluator.process_batch(data_claude)

print("\n--- FINAL EVALUATION REPORT for CLAUDE 4.5 SONNET OUTPUTS---")
print(json.dumps(final_report_claude, indent=2))

In [ ]:
data_qwen4b = load_json_file("outputs/responses/qwen3-4B/question_responses_new.json")
evaluator = Evaluator()
final_report_qwen4b = evaluator.process_batch(data_qwen4b["ModelGeneratedAnswersWithQuestions"])

print("\n--- FINAL EVALUATION REPORT for Qwen4 4B OUTPUTS (first 200 reviews) ---")
print(json.dumps(final_report_qwen4b, indent=2))